In [1]:
# Install sentence-transformers quietly
#!pip install -q sentence-transformers

import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from scipy.optimize import minimize
import os

# Verify Kaggle input directory
print("Available datasets in Kaggle:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Available datasets in Kaggle:
/kaggle/input/datasets/marwanemam/assistments-1/student_log_1.csv
/kaggle/input/datasets/marwanemam/assistments-full/anonymized_full_release_competition_dataset.csv
/kaggle/input/datasets/marwanemam/dbe-kt22/KCs.csv
/kaggle/input/datasets/marwanemam/dbe-kt22/Question_KC_Relationships.csv
/kaggle/input/datasets/marwanemam/dbe-kt22/Practice_Sequences.json


In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class PFATracker:
    def __init__(self, concepts, num_levels=6, top_k=5, alpha=0.1, beta_concept=None):
        self.concepts = concepts
        self.num_levels = num_levels
        self.top_k = top_k
        self.alpha = alpha

        self.num_concepts = len(concepts)
        self.idx = {c: i for i, c in enumerate(concepts)}

        self.successes = np.zeros((self.num_concepts, num_levels))
        self.failures = np.zeros((self.num_concepts, num_levels))
        self.propagation_bonus = np.zeros((self.num_concepts, num_levels))

        if beta_concept is None:
            rng = np.random.default_rng(42)
            self.beta_concept = rng.uniform(-0.3, 0.3, self.num_concepts)
        else:
            self.beta_concept = np.asarray(beta_concept, dtype=float)

        self.beta_level = np.array([0.0, -0.4, -0.9, -1.5, -2.1, -2.9])
        self.gamma_level = np.array([0.45, 0.40, 0.35, 0.30, 0.25, 0.2])
        self.rho_level = np.array([0.30, 0.35, 0.40, 0.45, 0.50, 0.60])

        self._build_similarity_graph()

    def _build_similarity_graph(self):
        model = SentenceTransformer("BAAI/bge-base-en-v1.5")
        embeddings = model.encode(self.concepts, normalize_embeddings=True)
        sim_matrix = embeddings @ embeddings.T  
        
        self.neighbors = []
        for i in range(self.num_concepts):
            sims = sim_matrix[i]
            idxs = np.argsort(-sims)[1:self.top_k + 1] 
            self.neighbors.append(idxs)
            
        self.sim_matrix = sim_matrix

    def predict(self, concept, level):
        i = self.idx[concept]
        k = level - 1

        z = (
            self.beta_concept[i]
            + self.beta_level[k]
            + self.gamma_level[k] * np.log1p(self.successes[i][k])
            - self.rho_level[k] * np.log1p(self.failures[i][k])
            + self.propagation_bonus[i][k]
        )

        if k > 0:
            s_lower = self.successes[i][k - 1]
            f_lower = self.failures[i][k - 1]
            lower_contribution = (
                self.gamma_level[k - 1] * np.log1p(s_lower)
                - self.rho_level[k - 1] * np.log1p(f_lower)
            )
            z += 0.25 * lower_contribution

        return sigmoid(z)

    def update(self, concept, level, correct):
        i = self.idx[concept]
        k = level - 1

        old_p = self.predict(concept, level)

        if correct:
            self.successes[i][k] += 1
        else:
            self.failures[i][k] += 1

        new_p = self.predict(concept, level)
        delta = np.clip(new_p - old_p, -0.1, 0.1)

        for j in self.neighbors[i]:
            sim = self.sim_matrix[i][j]
            for lvl in range(k + 1):
                self._propagate(j, lvl, delta, sim)

    def _propagate(self, j, lvl, delta, sim):
        weight = self.alpha * sim / (lvl + 1)  
        self.propagation_bonus[j][lvl] += weight * delta

In [4]:
# data_path = '/kaggle/input/datasets/marwanemam/assistments-full/anonymized_full_release_competition_dataset.csv' 

# print("Loading dataset...")
# df = pd.read_csv(data_path, low_memory=False)

# # Clean and sort chronologically using the specific ASSISTments column names
# df = df.dropna(subset=['skill', 'correct'])
# df['correct'] = df['correct'].astype(int)

# # Sort by student ID and Start Time
# df = df.sort_values(by=['studentId', 'startTime'])

# unique_concepts = df['skill'].unique().tolist()
# print(f"Dataset contains {len(unique_concepts)} unique concepts.")

# # Group by the correct student ID column
# trajectories = df.groupby('studentId')
# print(f"Grouped into {len(trajectories)} student trajectories.")

# # Initialize tracker
# tracker = PFATracker(concepts=unique_concepts, num_levels=6)

# X_features = []  
# y_targets = []   

# print("Extracting PFA state features...")

# # Process a sample of 2500 students to save memory/time. 
# for student_id, student_df in list(trajectories)[:2500]:
#     tracker.successes.fill(0)
#     tracker.failures.fill(0)
#     tracker.propagation_bonus.fill(0)
    
#     for _, row in student_df.iterrows():
#         concept = row['skill'] # Updated column name
#         correct = row['correct']
#         lvl = 1 # Defaulting ASSISTments to Bloom's Level 1
        
#         i = tracker.idx[concept]
#         k = lvl - 1
        
#         s_count = tracker.successes[i][k]
#         f_count = tracker.failures[i][k]
#         prop_bonus = tracker.propagation_bonus[i][k]
        
#         lower_contrib = 0.0
#         if k > 0:
#             s_lower = tracker.successes[i][k - 1]
#             f_lower = tracker.failures[i][k - 1]
#             lower_contrib = (tracker.gamma_level[k - 1] * np.log1p(s_lower) - 
#                              tracker.rho_level[k - 1] * np.log1p(f_lower))
            
#         X_features.append([i, k, np.log1p(s_count), np.log1p(f_count), lower_contrib, prop_bonus])
#         y_targets.append(correct)
        
#         tracker.update(concept, lvl, correct)

# X_features = np.array(X_features)
# y_targets = np.array(y_targets)
# print(f"Extraction complete. Shape: {X_features.shape}")

Loading dataset...
Dataset contains 102 unique concepts.
Grouped into 1709 student trajectories.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extracting PFA state features...
Extraction complete. Shape: (942816, 6)


In [5]:
# # Helper functions to pack/unpack parameters for the SciPy optimizer
# num_c = tracker.num_concepts
# num_l = tracker.num_levels

# def pack_params(tracker):
#     return np.concatenate([
#         tracker.beta_concept, 
#         tracker.beta_level, 
#         tracker.gamma_level, 
#         tracker.rho_level
#     ])

# def unpack_params(x, num_c, num_l):
#     b_c = x[0 : num_c]
#     b_l = x[num_c : num_c + num_l]
#     g_l = x[num_c + num_l : num_c + 2*num_l]
#     r_l = x[num_c + 2*num_l : num_c + 3*num_l]
#     return b_c, b_l, g_l, r_l

# # Highly vectorized Binary Cross-Entropy Loss function
# def pfa_loss(x, X_feat, y_true, num_c, num_l):
#     b_c, b_l, g_l, r_l = unpack_params(x, num_c, num_l)
    
#     i_idx = X_feat[:, 0].astype(int)
#     k_idx = X_feat[:, 1].astype(int)
#     log_s = X_feat[:, 2]
#     log_f = X_feat[:, 3]
#     lower_contrib = X_feat[:, 4]
#     prop_bonus = X_feat[:, 5]
    
#     # Calculate Z (logit) according to your exact formula
#     z = b_c[i_idx] + b_l[k_idx] + (g_l[k_idx] * log_s) - (r_l[k_idx] * log_f) + prop_bonus + (0.25 * lower_contrib)
    
#     # Sigmoid with clipping to prevent overflow warnings
#     p = 1 / (1 + np.exp(-np.clip(z, -15, 15)))
#     p = np.clip(p, 1e-7, 1 - 1e-7)
    
#     # Binary Cross Entropy Loss + mild L2 regularization to prevent overfitting
#     bce = -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
#     l2_reg = 0.0001 * np.sum(x**2) 
    
#     return bce + l2_reg

# # 1. Evaluate Initial Loss
# initial_params = pack_params(tracker)
# initial_loss = pfa_loss(initial_params, X_features, y_targets, num_c, num_l)
# print(f"Initial Log Loss (Untrained): {initial_loss:.4f}")

# # 2. Run the Optimizer
# print("Running L-BFGS-B Optimizer... (This may take a minute or two)")
# result = minimize(
#     fun=pfa_loss,
#     x0=initial_params,
#     args=(X_features, y_targets, num_c, num_l),
#     method='L-BFGS-B',
#     options={'maxiter': 100, 'disp': True}
# )

# # 3. Apply the optimized parameters back to the tracker
# opt_b_c, opt_b_l, opt_g_l, opt_r_l = unpack_params(result.x, num_c, num_l)

# tracker.beta_concept = opt_b_c
# tracker.beta_level = opt_b_l
# tracker.gamma_level = opt_g_l
# tracker.rho_level = opt_r_l

# final_loss = pfa_loss(result.x, X_features, y_targets, num_c, num_l)
# print(f"Final Log Loss (Trained): {final_loss:.4f}")
# print("SUCCESS: Tracker parameters have been automatically updated based on real ASSISTments data!")

# # Quick sanity check on the new global parameters:
# print(f"\nNew Gamma (Learning Rate) array: \n{np.round(tracker.gamma_level, 4)}")
# print(f"New Rho (Failure Penalty) array: \n{np.round(tracker.rho_level, 4)}")

Initial Log Loss (Untrained): 0.6860
Running L-BFGS-B Optimizer... (This may take a minute or two)


/tmp/ipykernel_57/3074106026.py:51: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


Final Log Loss (Trained): 0.6373
SUCCESS: Tracker parameters have been automatically updated based on real ASSISTments data!

New Gamma (Learning Rate) array: 
[4.713e-01 6.000e-04 5.000e-04 5.000e-04 4.000e-04 3.000e-04]
New Rho (Failure Penalty) array: 
[0.4637 0.0005 0.0006 0.0007 0.0007 0.0009]


In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import os

# ==========================================
# 1. Paths
# ==========================================
DATA_DIR = '/kaggle/input/datasets/marwanemam/dbe-kt22/'
json_path = os.path.join(DATA_DIR, 'Practice_Sequences.json')
q_kc_path = os.path.join(DATA_DIR, 'Question_KC_Relationships.csv')
kcs_path = os.path.join(DATA_DIR, 'KCs.csv')

# ==========================================
# 2. Map Question IDs to Concept Text
# ==========================================
print("Mapping Question IDs to Concepts...")
# Load the relationship and concept tables
df_q_kc = pd.read_csv(q_kc_path)
df_kcs = pd.read_csv(kcs_path)

# Merge to get the text name for each question_id
df_merged = df_q_kc.merge(df_kcs, left_on='knowledgecomponent_id', right_on='id')
q_to_concept = dict(zip(df_merged['question_id'].astype(str), df_merged['name']))

unique_concepts = list(q_to_concept.values())
print(f"Loaded {len(unique_concepts)} unique concepts.")

# ==========================================
# 3. Initialize Tracker & Map Difficulties
# ==========================================
tracker = PFATracker(concepts=unique_concepts, num_levels=6)

# Map DBE-KT22 ground truth difficulty (1,2,3) to PFA Bloom's levels (1-6)
diff_map = {
    '1': 1, # Easy -> Level 1
    '2': 3, # Medium -> Level 3
    '3': 5  # Hard -> Level 5
}

# ==========================================
# 4. Parse JSON & Extract Features
# ==========================================
print("Loading JSON trajectories and extracting features...")
with open(json_path, 'r') as f:
    trajectories = json.load(f)

X_features = []  
y_targets = []   

for seq in trajectories:
    tracker.successes.fill(0)
    tracker.failures.fill(0)
    tracker.propagation_bonus.fill(0)
    
    # Split the comma-separated strings into iterable lists
    qids = seq['question_ids'].split(',')
    answers = seq['answers'].split(',')
    difficulties = seq['gt_difficulty'].split(',')
    confidences = seq['answer_confidence'].split(',')
    hints = seq['hint_used'].split(',')
    
    # Walk through the sequence steps
    for qid, ans_str, diff_str, conf_str, hint_str in zip(qids, answers, difficulties, confidences, hints):
        
        concept = q_to_concept.get(qid)
        if not concept:
            continue # Skip if the question lacks a mapped concept
            
        correct = int(ans_str)
        lvl = diff_map.get(diff_str, 1) 
        
        i = tracker.idx[concept]
        k = lvl - 1
        
        # --- A. Extract State BEFORE Update ---
        s_count = tracker.successes[i][k]
        f_count = tracker.failures[i][k]
        prop_bonus = tracker.propagation_bonus[i][k]
        
        lower_contrib = 0.0
        if k > 0:
            s_lower = tracker.successes[i][k - 1]
            f_lower = tracker.failures[i][k - 1]
            lower_contrib = (tracker.gamma_level[k - 1] * np.log1p(s_lower) - 
                             tracker.rho_level[k - 1] * np.log1p(f_lower))
            
        X_features.append([i, k, np.log1p(s_count), np.log1p(f_count), lower_contrib, prop_bonus])
        y_targets.append(correct)
        
        # --- B. Update Tracker WITH Bias Reduction ---
        old_p = tracker.predict(concept, lvl)
        
        if correct == 1:
            # BIAS REDUCTION: If they used a hint or reported low confidence (1), penalize the success
            if hint_str.strip().lower() == 'true' or conf_str == '1':
                tracker.successes[i][k] += 0.4  # Only give partial credit for a guessed/hinted success
            else:
                tracker.successes[i][k] += 1.0
        else:
            tracker.failures[i][k] += 1.0

        # Propagate to neighbors
        new_p = tracker.predict(concept, lvl)
        delta = np.clip(new_p - old_p, -0.1, 0.1)
        for j in tracker.neighbors[i]:
            sim = tracker.sim_matrix[i][j]
            for curr_lvl in range(k + 1):
                tracker._propagate(j, curr_lvl, delta, sim)

X_features = np.array(X_features)
y_targets = np.array(y_targets)
print(f"Extraction complete. Shape: {X_features.shape}")

# ==========================================
# 5. The Optimizer (L-BFGS-B)
# ==========================================
num_c = tracker.num_concepts
num_l = tracker.num_levels

def pack_params(t):
    return np.concatenate([t.beta_concept, t.beta_level, t.gamma_level, t.rho_level])

def unpack_params(x, c, l):
    return x[0:c], x[c:c+l], x[c+l:c+2*l], x[c+2*l:c+3*l]

def pfa_loss(x, X_feat, y_true, c, l):
    b_c, b_l, g_l, r_l = unpack_params(x, c, l)
    
    i_idx = X_feat[:, 0].astype(int)
    k_idx = X_feat[:, 1].astype(int)
    log_s = X_feat[:, 2]
    log_f = X_feat[:, 3]
    lower_contrib = X_feat[:, 4]
    prop_bonus = X_feat[:, 5]
    
    z = b_c[i_idx] + b_l[k_idx] + (g_l[k_idx] * log_s) - (r_l[k_idx] * log_f) + prop_bonus + (0.25 * lower_contrib)
    p = 1 / (1 + np.exp(-np.clip(z, -15, 15)))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)) + (0.0001 * np.sum(x**2))

print("Running Optimizer...")
init_params = pack_params(tracker)
result = minimize(
    fun=pfa_loss,
    x0=init_params,
    args=(X_features, y_targets, num_c, num_l),
    method='L-BFGS-B',
    options={'maxiter': 100}
)

opt_b_c, opt_b_l, opt_g_l, opt_r_l = unpack_params(result.x, num_c, num_l)
tracker.beta_concept, tracker.beta_level = opt_b_c, opt_b_l
tracker.gamma_level, tracker.rho_level = opt_g_l, opt_r_l

print(f"Final Trained Loss: {result.fun:.4f}")
print("New Gamma (Learning Rates):", np.round(tracker.gamma_level, 4))
# 1. Print Rho (Failure Penalties)
print("\nNew Rho (Failure Penalties):")
print(np.round(tracker.rho_level, 4))

# 2. Print Beta Level (Base difficulty of each Bloom's Level)
print("\nNew Beta Levels (Base Difficulty):")
print(np.round(tracker.beta_level, 4))

# 3. Print a sample of Beta Concept (Base difficulty per specific concept)
print("\nSample of Beta Concepts (Inherent difficulty of topics):")
for i in range(min(10, tracker.num_concepts)): 
    concept_name = tracker.concepts[i]
    beta_val = tracker.beta_concept[i]
    print(f"  {concept_name}: {beta_val:.4f}")

Mapping Question IDs to Concepts...
Loaded 212 unique concepts.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading JSON trajectories and extracting features...
Extraction complete. Shape: (147726, 6)
Running Optimizer...
Final Trained Loss: 0.4875
New Gamma (Learning Rates): [ 3.300e-02 -4.000e-04  1.494e-01 -3.000e-04  8.884e-01 -2.000e-04]

New Rho (Failure Penalties):
[ 1.444e-01 -3.000e-04  3.433e-01 -4.000e-04  2.331e-01 -6.000e-04]

New Beta Levels (Base Difficulty):
[1.4333e+00 4.0000e-04 1.0389e+00 1.4000e-03 4.2710e-01 2.6000e-03]

Sample of Beta Concepts (Inherent difficulty of topics):
  Data type: -0.0002
  Attribute: 0.0000
  Relational data model: -0.0002
  Relational data model: -0.0001
  Superkey: 0.0002
  Foreign key: -0.0003
  Superkey: -0.0001
  Subset: -0.0002
  SELECT: 0.0002
  Schema: 0.0000
